# Symbolic generation of kinetic integral code

## Kinetic-Energy Integrals over Cartesian Gaussians

After finishing the overlap integrals we now switch to kinetic energy integrals, which are almost as easy to calculate analytically as the overlaps.
The matrix elements of the kinetic energy operator over Cartesian Gaussians are defined as:

$$
T_{\mu\nu} \;=\; \left\langle \chi_\mu \middle| -\frac{1}{2}\nabla^2 \middle| \chi_\nu \right\rangle
\;=\;
-\frac{1}{2}\int \chi_\mu(\mathbf r)\,\nabla^2 \chi_\nu(\mathbf r)\,d\mathbf r.
$$

A Cartesian Gaussian on center $\mathbf B$ has a general form:
$$
\chi_{lmn}(\mathbf r;\beta,\mathbf B)
=
(x-B_x)^l (y-B_y)^m (z-B_z)^n
\,e^{-\beta |\mathbf r-\mathbf B|^2}.
$$

that can be factorized. By defining one dimensional factors like 

$$
g_l(x)= (x-B_x)^l e^{-\beta(x-B_x)^2}.
$$

the second derivative can be calculated as: 

$$
\frac{d^2 g_l}{dx^2}
=
\left[
l(l-1)(x-B_x)^{l-2}
-2\beta(2l+1)(x-B_x)^l
+4\beta^2(x-B_x)^{l+2}
\right]e^{-\beta(x-B_x)^2}.
$$

Therefore
$$
-\frac{1}{2}\frac{d^2 g_l}{dx^2}
=
-\frac{1}{2}l(l-1)g_{l-2}
+\beta(2l+1)g_l
-2\beta^2 g_{l+2}.
$$

This expression allows us to express the kinetic energy integrals in terms of the overlaps that we already know how to calculate.

We begin by calculating analytically the overlap between two primitive one dimensional s-type Gaussian functions, with the exponents $\alpha$ and $\beta$ and centers at $Ax$ and $Bx$ respectively. The overlap integral can be directly evaluated using the following sympy code


In [ ]:
import sympy as sp

In [ ]:
x, y, z = sp.symbols('x y z', real=True)
Ax, Ay, Az, Bx, By, Bz = sp.symbols('Ax Ay Az Bx By Bz', real=True)
alpha, beta = sp.symbols('alpha beta', real=True, positive=True)

In [ ]:
g1x = sp.exp(-alpha * (x - Ax)**2) 
g1y = sp.exp(-alpha * (y - Ay)**2)
g1z = sp.exp(-alpha * (z - Az)**2)
g2x = sp.exp(-beta * (x - Bx)**2)
g2y = sp.exp(-beta * (y - By)**2)
g2z = sp.exp(-beta * (z - Bz)**2)
T00x = sp.integrate((sp.simplify(g1x.diff(x,1) * g2x.diff(x,1))), (x, -sp.oo, sp.oo))*sp.integrate(g1y * g2y, (y, -sp.oo, sp.oo)) * sp.integrate(g1z * g2z, (z, -sp.oo, sp.oo))
T00y = sp.integrate((sp.simplify(g1y.diff(y,1) * g2y.diff(y,1))), (y, -sp.oo, sp.oo))*sp.integrate(g1x * g2x, (x, -sp.oo, sp.oo)) * sp.integrate(g1z * g2z, (z, -sp.oo, sp.oo))
T00z = sp.integrate((sp.simplify(g1z.diff(z,1) * g2z.diff(z,1))), (z, -sp.oo, sp.oo))*sp.integrate(g1x * g2x, (x, -sp.oo, sp.oo)) * sp.integrate(g1y * g2y, (y, -sp.oo, sp.oo)) 
T00 = 1/2 * (T00x + T00y + T00z)
T00 = sp.simplify(T00)
T00

In [ ]:
T00 = sp.simplify(
    (alpha * beta / (alpha + beta))
    * (
        3
        - 2 * (alpha * beta / (alpha + beta))
        * ((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)
    )
    * (sp.pi / (alpha + beta))**sp.Rational(3, 2)
    * sp.exp(
        -(alpha * beta / (alpha + beta))
        * ((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)
    )
)
T00

We will need the recursive function for the Cartesian-to-Hermite coefficients, which we have already implemented and tested in `gaussian_functions.ipynb`. 

In [ ]:
from functools import lru_cache
@lru_cache(maxsize=None)
def get_ckn(k: int, n: int, p):
    """Coefficient C_k^n in the Cartesian-to-Hermite expansion."""
    if k < 0 or k > n:
        return sp.Integer(0)

    if n == 0:
        return sp.Integer(1) if k == 0 else sp.Integer(0)

    return get_ckn(k - 1, n - 1, p) / (2 * p) + (k + 1) * get_ckn(k + 1, n - 1, p)

## Strategy for symbolic generation of kinetic energy formulas

In order to generate the formulas for the kinetic energy integrals, we will implement the operator form of the kinetic energy integral as described above. We will use sympy to evaluate the required derivatives and simplify the resulting expressions. The derivatives up to some predefined maximum angular momentum $L_{\max}$ will be evaluated and stored in a dictionary for efficient access.  As a helper function we will implement a function that generates the required index tuples $(i,j,k,l,m,n)$ for the Cartesian Gaussians up to $L_{\max}$.

In [ ]:
from functools import lru_cache
def build_derivatives_table(base_integral, Lmax, Avars, Bvars):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars

    all_idx = [
        (i, j, L - i - j)
        for L in range(Lmax + 1)
        for i in range(L + 1)
        for j in range(L + 1 - i)
    ]

    @lru_cache(maxsize=None)
    def derivative(i, j, k, l, m, n):
        if (i, j, k, l, m, n) == (0, 0, 0, 0, 0, 0):
            return base_integral

        if i > 0:
            return sp.diff(derivative(i - 1, j, k, l, m, n), Ax)
        if j > 0:
            return sp.diff(derivative(i, j - 1, k, l, m, n), Ay)
        if k > 0:
            return sp.diff(derivative(i, j, k - 1, l, m, n), Az)
        if l > 0:
            return sp.diff(derivative(i, j, k, l - 1, m, n), Bx)
        if m > 0:
            return sp.diff(derivative(i, j, k, l, m - 1, n), By)
        return sp.diff(derivative(i, j, k, l, m, n - 1), Bz)

    derivatives_dict = {}
    for (i, j, k) in all_idx:
        for (l, m, n) in all_idx:
            derivatives_dict[(i, j, k, l, m, n)] = sp.simplify(derivative(i, j, k, l, m, n))     

    return derivatives_dict

## How does the `build_derivatives_table` function work?

The `build_derivatives_table` function takes as input the base integral, the maximum angular momentum, and the variables corresponding to the centers of the Gaussians. It generates all required index tuples for the Cartesian Gaussians up to $L_{\max}$, evaluates the derivatives according to the operator form of the overlap integral, simplifies them using sympy, and stores them in a dictionary for efficient access during code subsequent generation. The derivatives are calculated recursively in the function named `derivative`, that reduces each operator to a base case by lowering the indices until it reaches the base integral $S_{000000}$, which is evaluated directly. lru_cache is used to optimize the recursive calls by caching previously computed results. In the following we call the function with Lmax=3, which means that we will generate formulas for all Cartesian Gaussians up to f-type (angular momentum 3).

In [ ]:
Lmax = 2

In [ ]:
derivatives_dict = build_derivatives_table(T00, Lmax, (Ax, Ay, Az), (Bx, By, Bz))

In [ ]:
def get_integral_expressions(integral_indices, derivatives_dict, alpha, beta):
    integral_expressions = {}

    for (i,j,k,l,m,n) in integral_indices:

        ci = [get_ckn(o, i, alpha) for o in range(i + 1)]
        cj = [get_ckn(p, j, alpha) for p in range(j + 1)]
        ck = [get_ckn(q, k, alpha) for q in range(k + 1)]

        cl = [get_ckn(r, l, beta) for r in range(l + 1)]
        cm = [get_ckn(s, m, beta) for s in range(m + 1)]
        cn = [get_ckn(t, n, beta) for t in range(n + 1)]

        expr = 0
        for o, co in enumerate(ci):
            for p, cp in enumerate(cj):
                for q, cq in enumerate(ck):
                    for r, cr in enumerate(cl):
                        for s, cs in enumerate(cm):
                            for t, ct in enumerate(cn):
                                expr += (
                                    co * cp * cq * cr * cs * ct
                                    * derivatives_dict[(o, p, q, r, s, t)]
                                )

        integral_expressions[(i,j,k,l,m,n)] = sp.simplify(expr)
    return integral_expressions

## How does the function `get_integral_expressions` work?

This function constructs symbolic Cartesian kinetic energy integral formulas for a set of angular-momentum index tuples that is passed as an argument. For each target integral index

$$
(i,j,k,l,m,n),      
$$

it builds the Hermite-expansion coefficients for bra and ket directions:

$$
\{C_o^{\,i}\},\{C_p^{\,j}\},\{C_q^{\,k}\},\{C_r^{\,l}\},\{C_s^{\,m}\},\{C_t^{\,n}\}.
$$

These are generated using recursive function `get_ckn(...)` for the corresponding Gaussian exponents `alpha` (bra) and `beta` (ket). It then evaluates the full six-dimensional contraction

$$
T_{ijk,lmn}
=
\sum_{o=0}^{i}\sum_{p=0}^{j}\sum_{q=0}^{k}
\sum_{r=0}^{l}\sum_{s=0}^{m}\sum_{t=0}^{n}
C_o^{\,i}C_p^{\,j}C_q^{\,k}C_r^{\,l}C_s^{\,m}C_t^{\,n}\,
D_{opqrst},
$$

where

$$
D_{opqrst} = \texttt{derivatives\_dict[(o,p,q,r,s,t)]}
$$

is the precomputed symbolic derivative of the base overlap expression.

The resulting symbolic expression is simplified with `sp.simplify(...)` and stored in a dictionary under key `(i,j,k,l,m,n)`.

### Which indices are generated?

We will generate all possible index tuples $(i,j,k,l,m,n)$ for Cartesian Gaussians up to angular momentum $L_{\max}=3$, which means that we will generate formulas for all s-, p-, d-, and f-type Cartesian Gaussians. This is not the most efficient approach since it ignores the symmetry and is redundant since many of the integrals are related by permutation of arguments. The length of the generated code could be reduced by generating only the unique integrals and then applying symmetry relations to obtain the rest. However, for simplicity and clarity we will generate all integrals explicitly. Later on this could be optimized by generating only the unique integrals and applying symmetry relations to obtain the rest. 

We implement a helper function l_to_ijk(L) that generates the index tuples for a given value of L, and then we use it to generate all index tuples up to Lmax.

In [ ]:
def l_to_ijk(L):
    IJK = []
    for I in range(L, -1, -1):
        for J in range(L - I, -1, -1):
            IJK.append((I, J, L - I - J))
    return sorted(IJK, reverse=True)

integral_indices=[]
ijk = [t for L in range(Lmax+1) for t in l_to_ijk(L)]
for i in ijk:
    for j in ijk:
        integral_indices.append(i+j)
integral_indices


In [ ]:
integral_expressions = get_integral_expressions(integral_indices, derivatives_dict, alpha, beta)

### Simplify expressions using substitutions 

In [ ]:
Dx, Dy, Dz, D2, AB_sum, AB_product = sp.symbols(
    'Dx Dy Dz D2 AB_sum AB_product',
    real=True,
)

subsdict = {
    Ax - Bx: Dx,
    Ay - By: Dy,
    Az - Bz: Dz,
    (Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2: D2,
    alpha + beta: AB_sum,
    alpha * beta: AB_product,
}

In [ ]:
for key, value in integral_expressions.items():
    integral_expressions[key] = sp.simplify(value.subs(subsdict, simultaneous=True))
#for key, value in integral_expressions.items():
#    print(f"Integral {key}: {value}")

## Generate code for the overlap integrals

In [ ]:
from pathlib import Path
from sympy.printing.numpy import NumPyPrinter, _known_functions_numpy, _known_constants_numpy

class TheochemNumPyPrinter(NumPyPrinter):
    def print_boys(self, expr):
        n, t = expr.args
        return f"boys({self._print(n)}, {self._print(t)})"

printer = TheochemNumPyPrinter()
printer._module = "np"
printer.known_functions = {k: f"np.{v}" for k, v in _known_functions_numpy.items()}
printer.known_functions["boys"] = "boys"
printer.known_constants = {k: f"np.{v}" for k, v in _known_constants_numpy.items()}

In [ ]:
def write_onel_module(path, name="S", use_cse=False, integral_expressions=None):
    lines = ["import numpy as np", 
             "from numba import njit",
             "@njit(cache=True, fastmath=True)",
             f"def {name}(i, j, k, l, m, n, Dx, Dy, Dz, D2, AB_sum, AB_product,alpha, beta):"]

    for key, value in integral_expressions.items():
        lines.append(f"    if (i, j, k, l, m, n) == {key}:")
        if use_cse:
            repls, reduced = sp.cse(value, symbols=sp.numbered_symbols("t"))
            for sym, expr in repls:
                lines.append(f"        {sym} = {printer.doprint(expr)}")
            lines.append(f"        return {printer.doprint(reduced[0])}")
        else:
            lines.append(f"        return {printer.doprint(value)}")

    lines.append("    raise KeyError((i, j, k, l, m, n))")

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
  
write_onel_module((Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent) / "src" / "theochem2026" / "integrals" / "T.py", name="T", use_cse = False, integral_expressions=integral_expressions)